# Plot examples from the Quplets paper

This notebook reproduces the five plotted examples included by `main.typ` for

$$F_k(x)=(1-k)\cos(2\pi n x)+k\cos(2\pi d x).$$

It consolidates the original scripts [`crests.py`](scripts/crests.py), [`Fk(x).py`](scripts/Fk(x).py), [`Quplet.py`](scripts/Quplet.py), and [`A.py`](scripts/A.py). The pitchfork is included as the even-$n$ case of the crest-trajectory calculation. The notebook uses bracketed root solving inside the paper's nearest-grid corridors, so no GUI prompts, symmetry-breaking perturbation, or peak-resolution tuning is required.

| Paper section | Example | Paper asset | Original script |
|---|---:|---|---|
| §2 | Crest positions $(n,d)=(5,7)$ | `Crests.svg` | `crests.py` |
| §3 | Balanced threshold profile $(n,d)=(5,8)$ | `Fk(x).svg` | `Fk(x).py` |
| §4 | Spacing vector $(n,d)=(53,67)$ | `Spacing Vector.svg` | `Quplet.py` |
| §6 | Crest amplitudes $(n,d)=(5,7)$ | `amplitude-trajectories-n5-d7.svg` | `A.py` |
| §7 | Central pitchfork $(n,d)=(4,7)$ | `pitchfork.svg` | `crests.py` |

In [ ]:
from math import gcd
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import brentq

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Set this to True to regenerate the five SVG files used by main.typ.
EXPORT_FIGURES = False

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "main.typ").exists() and (PROJECT_ROOT.parent / "main.typ").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
FIGURES_DIR = PROJECT_ROOT / "Figures"

def maybe_save(fig, filename):
    if EXPORT_FIGURES:
        FIGURES_DIR.mkdir(exist_ok=True)
        fig.savefig(FIGURES_DIR / filename, format="svg", bbox_inches="tight", pad_inches=0.05)

## Numerical model

A stationary point satisfies $G_k(x)=0$, where the irrelevant factor $-2\pi$ has been removed from $F'_k$. For each non-tied anchor $i/n$, the paper proves that the anchored crest stays in the short interval joining $i/n$ to its nearest $d$-grid point. `brentq` solves for the unique root in that corridor.

In [ ]:
def F(x, k, n, d):
    """The two-frequency cosine family F_k(x)."""
    return (1.0 - k) * np.cos(2 * np.pi * n * x) + k * np.cos(2 * np.pi * d * x)


def G(x, k, n, d):
    """Stationarity equation F'_k(x)=0, with the factor -2π removed."""
    return (1.0 - k) * n * np.sin(2 * np.pi * n * x) + k * d * np.sin(2 * np.pi * d * x)


def crest_indicator(x, k, n, d):
    """Positive exactly when F''_k(x)<0."""
    return (1.0 - k) * n**2 * np.cos(2 * np.pi * n * x) + k * d**2 * np.cos(2 * np.pi * d * x)


def nearest_integer(z):
    """Paper convention: choose the upper integer at a half-integer tie."""
    return int(np.floor(z + 0.5))


def parameter_grid(*special_values, points=501):
    """A uniform [0,1] grid containing all requested exact parameters."""
    return np.unique(np.r_[np.linspace(0.0, 1.0, points), special_values])


def anchored_position(n, d, anchor_index, k):
    """Solve one non-tied anchored branch in its nearest-grid corridor."""
    anchor = anchor_index / n
    endpoint = nearest_integer(d * anchor_index / n) / d

    if np.isclose(anchor, endpoint, atol=1e-15):
        return anchor
    if k <= 0.0:
        return anchor
    if k >= 1.0:
        return endpoint

    left, right = sorted((anchor, endpoint))
    return brentq(G, left, right, args=(k, n, d), xtol=2e-14, rtol=1e-14)


def anchored_branches(n, d, k_values, *, skip_indices=()):
    """Return lifted branches i=0,…,n; the first and last are one circular branch."""
    if not (1 < n < d) or gcd(n, d) != 1:
        raise ValueError("Expected coprime integers satisfying 1 < n < d.")

    skipped = set(skip_indices)
    positions = np.full((len(k_values), n + 1), np.nan)
    for i in range(n + 1):
        if i in skipped:
            continue
        positions[:, i] = [anchored_position(n, d, i, k) for k in k_values]
    return positions


def fraction_ticks(denominator):
    values = np.linspace(0.0, 1.0, denominator + 1)
    labels = [r"$0$"] + [rf"$\frac{{{i}}}{{{denominator}}}$" for i in range(1, denominator)] + [r"$1$"]
    return values, labels

## 1. Anchored crest-position trajectories — §2

For $(n,d)=(5,7)$, the five branches start on the regular $5$-grid and end at their uniquely selected nearest sites on the regular $7$-grid. The top and bottom curves are the same fixed circular branch, shown in a lifted interval.

In [ ]:
n, d = 5, 7
k_values = parameter_grid(points=501)
positions_5_7 = anchored_branches(n, d, k_values)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.0, 1.0, n + 1))
for i in range(n + 1):
    ax.plot(k_values, positions_5_7[:, i], color=colors[i], linewidth=1.6)

ax.set(xlim=(0, 1), ylim=(0, 1), xlabel=r"$k$", ylabel=r"$X_i$", title=rf"$X_{{i,{n},{d}}}(k)$")
ax.set_xticks([0.0, 1.0], [r"$0$", r"$1$"])
left_ticks, left_labels = fraction_ticks(n)
ax.set_yticks(left_ticks, left_labels)
right_ax = ax.secondary_yaxis("right")
right_ticks, right_labels = fraction_ticks(d)
right_ax.set_yticks(right_ticks, right_labels)
ax.tick_params(axis="both", labelsize=13)
right_ax.tick_params(axis="y", labelsize=13)
fig.tight_layout()
maybe_save(fig, "Crests.svg")
plt.show()

## 2. Balanced threshold profile — §3

For $(n,d)=(5,8)$, the balanced value is $k_c=n/(n+d)=5/13$. The gray grading is the resonant $(n+d)$-grid. Red points mark the five highest crests (with the fixed crest repeated at $x=1$ to close the plotted period).

In [ ]:
n, d = 5, 8
N = n + d
k_c = n / N
x_values = np.linspace(0.0, 1.0, 4001)
profile = F(x_values, k_c, n, d)
selected_indices = np.array([nearest_integer(i * N / n) for i in range(n + 1)])
selected_x = selected_indices / N

fig, ax = plt.subplots(figsize=(10, 5))
for j in range(N + 1):
    grid_x = j / N
    ax.axvline(grid_x, color="0.86", linestyle="--", linewidth=0.8, zorder=0)
    ax.text(grid_x, -0.04, rf"$\frac{{{j}}}{{{N}}}$", transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=8, clip_on=False)
ax.plot(x_values, profile, color="tab:blue", linewidth=1.6)
ax.scatter(selected_x, F(selected_x, k_c, n, d), color="tab:red", s=30, zorder=3)
ax.set(xlim=(0, 1), xlabel=r"$x$", ylabel=r"$F_{5/13}(x)$")
ax.set_xticks([])
ax.set_yticks([])
fig.tight_layout()
maybe_save(fig, "Fk(x).svg")
plt.show()

## 3. Huplet spacing-vector trajectory — §4

The circular spacing vector is $\Delta_i(k)=X_{i+1}(k)-X_i(k)$. Reflection symmetry leaves 27 displayed trajectories for $(n,d)=(53,67)$. At $k_c=53/120$ and $k=1$, the components lock to the two grid values described in the paper.

In [ ]:
n, d = 53, 67
k_c = n / (n + d)
k_values = parameter_grid(k_c, points=401)
positions_53_67 = anchored_branches(n, d, k_values)
spacings_53_67 = np.diff(positions_53_67, axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
displayed_count = n // 2 + 1
colors = plt.cm.viridis(np.linspace(0.0, 1.0, displayed_count))
for i in range(displayed_count):
    ax.plot(k_values, spacings_53_67[:, i], color=colors[i], linewidth=1.0)

ax.set(xlim=(0, 1), xlabel=r"$k$", ylabel=r"$\Delta X_i$", title=rf"Components of $\Delta X_{{i,{n},{d}}}(k)$")
ax.set_xticks([0.0, k_c, 1.0], [r"$0$", rf"$\frac{{{n}}}{{{n + d}}}$", r"$1$"])
ax.set_yticks([1 / d, 1 / n, 2 / d], [rf"$\frac{{1}}{{{d}}}$", rf"$\frac{{1}}{{{n}}}$", rf"$\frac{{2}}{{{d}}}$"])
ax.tick_params(axis="both", labelsize=13)
fig.tight_layout()
maybe_save(fig, "Spacing Vector.svg")
plt.show()

## 4. Anchored crest amplitudes — §6

The amplitude of branch $i$ is $A_i(k)=F_k(X_i(k))$. For $(n,d)=(5,7)$, reflection gives $A_4=A_1$ and $A_3=A_2$, so only three curves remain. Every displayed branch reaches its minimum at $k_c=5/12$.

In [ ]:
n, d = 5, 7
k_c = n / (n + d)
k_values = parameter_grid(k_c, points=501)
positions_5_7 = anchored_branches(n, d, k_values)
amplitudes_5_7 = F(positions_5_7.T, k_values, n, d)
critical_index = np.flatnonzero(np.isclose(k_values, k_c))[0]

fig, ax = plt.subplots(figsize=(8.2, 4.8))
colors = plt.cm.viridis(np.linspace(0.0, 1.0, 3))
for i in range(3):
    ax.plot(k_values, amplitudes_5_7[i], color=colors[i], linewidth=1.8, label=rf"$A_{i}(k)$")
    if i > 0:
        ax.scatter([k_c], [amplitudes_5_7[i, critical_index]], color=colors[i], s=24, zorder=4)

ax.axvline(k_c, color="black", linestyle="--", linewidth=0.9)
ax.annotate(r"$k_c$", xy=(k_c, 0.0), xycoords=ax.get_xaxis_transform(), xytext=(0, -3), textcoords="offset points", ha="center", va="top", annotation_clip=False)
ax.set(xlim=(0, 1), xlabel=r"$k$", ylabel=r"$A_i(k)$")
ax.set_xticks([0.0, 1.0], [r"$0.0$", r"$1.0$"])
ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.01, 0.55))
fig.tight_layout(rect=(0.0, 0.0, 0.82, 1.0))
maybe_save(fig, "amplitude-trajectories-n5-d7.svg")
plt.show()

## 5. Even-$n$ central pitchfork — §7

For $(n,d)=(4,7)$, the central branch $x=1/2$ loses crest status at

$$k_*=\frac{n^2}{n^2+d^2}=\frac{16}{65}.$$

The two off-center crest arms are found without a perturbation by removing the persistent central root from the stationarity equation: solve $G_k(x)/(x-1/2)=0$ on the two adjacent corridors.

In [ ]:
def central_pitchfork_arms(n, d, k_values):
    if n % 2 or d % 2 != 1:
        raise ValueError("This central pitchfork requires even n and odd d.")

    center = 0.5
    k_star = n**2 / (n**2 + d**2)
    left_endpoint = (d - 1) / (2 * d)
    right_endpoint = (d + 1) / (2 * d)
    left_arm = np.full_like(k_values, center, dtype=float)
    right_arm = np.full_like(k_values, center, dtype=float)

    def deflated_stationarity(x, k):
        if np.isclose(x, center, atol=1e-14):
            return 2 * np.pi * ((1 - k) * n**2 - k * d**2)
        return G(x, k, n, d) / (x - center)

    for j, k in enumerate(k_values):
        if k <= k_star:
            continue
        if k >= 1.0:
            left_arm[j], right_arm[j] = left_endpoint, right_endpoint
            continue
        left_arm[j] = brentq(deflated_stationarity, left_endpoint, center, args=(k,), xtol=2e-14, rtol=1e-14)
        right_arm[j] = brentq(deflated_stationarity, center, right_endpoint, args=(k,), xtol=2e-14, rtol=1e-14)

    return k_star, left_arm, right_arm


n, d = 4, 7
k_star = n**2 / (n**2 + d**2)
k_values = parameter_grid(k_star, points=501)
noncentral = anchored_branches(n, d, k_values, skip_indices={n // 2})
k_star, left_arm, right_arm = central_pitchfork_arms(n, d, k_values)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.0, 1.0, n + 1))
for i in range(n + 1):
    if i != n // 2:
        ax.plot(k_values, noncentral[:, i], color=colors[i], linewidth=1.6)
ax.plot(k_values, left_arm, color=colors[n // 2], linewidth=1.6)
ax.plot(k_values, right_arm, color=colors[n // 2], linewidth=1.6)
ax.axvline(k_star, color="gray", linewidth=1.0, linestyle="--")

ax.set(xlim=(0, 1), ylim=(0, 1), xlabel=r"$k$", ylabel=r"$X_i$", title=rf"$X_{{i,{n},{d}}}(k)$")
ax.set_xticks([0.0, k_star, 1.0], [r"$0$", rf"$\frac{{{n**2}}}{{{n**2 + d**2}}}$", r"$1$"])
left_ticks, left_labels = fraction_ticks(n)
ax.set_yticks(left_ticks, left_labels)
right_ax = ax.secondary_yaxis("right")
right_ticks, right_labels = fraction_ticks(d)
right_ax.set_yticks(right_ticks, right_labels)
ax.tick_params(axis="both", labelsize=13)
right_ax.tick_params(axis="y", labelsize=13)
fig.tight_layout()
maybe_save(fig, "pitchfork.svg")
plt.show()

## Reproducibility checks

These checks verify the exact grid-locking values highlighted in the paper and confirm that the two pitchfork arms end at the reflected $7$-grid pair.

In [ ]:
# Threshold crests lie on the 13-grid.
assert np.allclose(selected_x * 13, np.round(selected_x * 13))

# The spacing plot contains the two claimed values at both locking instants.
threshold_row = spacings_53_67[np.flatnonzero(np.isclose(parameter_grid(53 / 120, points=401), 53 / 120))[0]]
endpoint_row = spacings_53_67[-1]
assert set(np.round(threshold_row * 120).astype(int)) == {2, 3}
assert set(np.round(endpoint_row * 67).astype(int)) == {1, 2}

# The reflected pitchfork endpoints are 3/7 and 4/7.
assert np.isclose(left_arm[-1], 3 / 7)
assert np.isclose(right_arm[-1], 4 / 7)

print("All paper-figure checks passed.")